# Classical Stefan Equation for Permafrost Active Layer Thickness (ALT)
## Классическое уравнение Стефана для расчета глубины сезонного протаивания (ALT)

This notebook contains the physical formulation, Python implementation, and empirical calibration for the **Classical Stefan Model**, which calculates the thawing depth of permafrost ground based on annual heat input.

Этот блокнот содержит физическую формулировку, реализацию на Python и эмпирическую калибровку для **Классической модели Стефана**, которая вычисляет глубину сезонного таяния вечной мерзлоты на основе годового поступления тепла.

### 1. Mathematical Formulation / Математическая формулировка

The Stefan equation is derived from the one-dimensional heat conduction equation with a phase transition boundary (the Stefan problem). Under the assumption of a linear temperature profile in the thawed soil layer, the thawing depth ($ALT$) is expressed as:

Уравнение Стефана выводится из одномерного уравнения теплопроводности с фазовым переходом на границе (задача Стефана). В предположении о линейном профиле температуры в талом слое грунта, глубина сезонного протаивания ($ALT$) выражается как:

$$ALT = \sqrt{\frac{2 \cdot \lambda_t \cdot DDT}{\rho \cdot L}}$$

Where / Где:
- $\lambda_t$: Thermal conductivity of thawed soil / Теплопроводность талого грунта ($W/(m \cdot K)$).
- $DDT$: Degree-days of thawing / Сумма градусо-суток таяния ($\sum T > 0^\circ C$).
- $\rho$: Dry bulk density of soil / Плотность сухого скелета грунта ($kg/m^3$).
- $L$: Latent heat of fusion of water / Удельная теплота плавления льда ($J/kg$).

In practice, for simplified engineering calculations, we aggregate the physical constants into a single calibration coefficient $E$:
На практике для инженерных расчетов физические константы объединяют в единый калибровочный коэффициент $E$:

$$ALT = E \cdot \sqrt{DDT}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

print("Libraries successfully imported. / Библиотеки успешно импортированы.")

### 2. Python Implementation / Реализация модели на Python

Let's define the predictive function and its standard calibration parameters.

In [ ]:
def stefan_classic(ddt, E=4.3448):
    """
    Calculates active layer thickness (ALT) in cm based on DDT.
    Вычисляет глубину сезонного таяния (ALT) в см на основе DDT.
    """
    return E * np.sqrt(ddt)

# Standard baseline calculation / Базовый расчет
ddt_test = 2150  # Average DDT in Yakutsk / Среднее значение DDT для Якутска
alt_predicted = stefan_classic(ddt_test)
print(f"For DDT = {ddt_test} °C-days, predicted ALT = {alt_predicted:.2f} cm")
print(f"При DDT = {ddt_test} °С-сут, расчетная глубина ALT = {alt_predicted:.2f} см")

### 3. Empirical Calibration / Эмпирическая калибровка коэффициента E

Let's calibrate the model using actual measurements in Central Yakutia.

In [ ]:
# Historical observations (Yakutsk area examples) / Исторические данные для района Якутска
ddt_history = np.array([1980, 2050, 2150, 2220, 2300, 2350])
alt_measured = np.array([193.5, 196.2, 201.5, 204.8, 208.1, 210.5])

# Calibrate / Нахождение оптимального параметра E
popt, pcov = curve_fit(stefan_classic, ddt_history, alt_measured, p0=[4.0])
E_calibrated = popt[0]

print(f"Calibrated Stefan Coefficient (E) = {E_calibrated:.5f}")
print(f"Откалиброванный коэффициент Стефана (E) = {E_calibrated:.5f}")

### 4. Visualizing Results / Визуализация результатов

In [ ]:
ddt_range = np.linspace(1800, 2500, 200)
alt_fitted = stefan_classic(ddt_range, E=E_calibrated)

plt.figure(figsize=(10, 6))
plt.plot(ddt_range, alt_fitted, '-', color='#ef4444', label=f'Stefan Model (E = {E_calibrated:.4f})')
plt.scatter(ddt_history, alt_measured, color='#22d3ee', s=80, zorder=5, label='Measured Data / Фактические замеры')
plt.title('Stefan Model Fitting vs Yakutsk Observations\nКалибровка уравнения Стефана по наблюдениям Якутска')
plt.xlabel('Degree-Days of Thawing (DDT) / Градусо-сутки таяния (°C·сут)')
plt.ylabel('Active Layer Thickness (ALT) / Глубина протаивания (см)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.show()